# 02 — Preprocessing & Feature Engineering

**Course:** Machine Learning and Deep Learning (CBS, Spring 2026)
**Companion to:** `gameplan.md`, `01_eda_andreja.ipynb`

This notebook turns the raw ESS11 CSV into two clean, model-ready feature sets:

- **Feature Set A — raw (~22 features).** Every variable individually, sentinels removed, imputed, and lightly transformed (reverse-code `polintr`, add `lr_extreme = |lrscale - 5|`).
- **Feature Set B — composite (~15 features).** Same as A, but the four trust items become `trust_index`, the three efficacy items become `efficacy_index`, and the three satisfaction items become `satisfaction_index`. This is the "interpretable narrative" version we'll defend at the oral exam.

Both sets are saved to disk as CSV files. The modeling notebook (`03_modeling_andreja.ipynb`) loads them directly.

**Cleaning rules per variable come from the EDA decision table in §12 of `01_eda_andreja.ipynb`.**

---

### Note: media & news variables dropped

The original variable selection included `nwspol` (minutes/day on news/politics) and `netusoft` (frequency of internet use). A quantitative check showed that:

- `netusoft` correlates with `vote` at only r = +0.046 and is heavily redundant with age (r = −0.50) and education (r = +0.40), which are already in the model.
- `nwspol_log` correlates with `vote` at r = +0.149, but most of that signal is mediated through `polintr` (r = 0.34 between the two).
- A 5-fold cross-validated logistic regression PR-AUC drops by only **0.001** when both variables are removed (0.9296 → 0.9286 on Set A; 0.9295 → 0.9285 on Set B) — well within CV noise.

Both variables are therefore excluded from the final feature sets. The number of categories in the conceptual framework drops from 7 to 6.


## 0. Setup


In [1]:
import numpy as np
import pandas as pd
import warnings

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

DATA_PATH = 'ESS11e04_1.csv'
TARGET = 'vote'
print('Setup complete.')


Setup complete.


## 1. Load the raw data

Load only the columns we need: 22 features + target + 2 design weights = 25 columns.
The weights are kept here for completeness but are **not** used as model features.


In [2]:
USECOLS = [
    'vote',
    # 1. socio-demographic
    'agea', 'gndr', 'cntry',
    # 2. socio-economic
    'eduyrs', 'hinctnta',
    # 3. political interest & ideology
    'polintr', 'lrscale', 'clsprty',
    # 4. institutional trust
    'trstprl', 'trstplt', 'trstprt', 'trstlgl',
    # 5. civic & political engagement
    'contplt', 'sgnptit', 'cptppola', 'psppsgva', 'psppipla',
    # 6. satisfaction with system
    'stfdem', 'stfgov', 'stfeco',
    # weights (descriptive only, not used as features)
    'pspwght', 'anweight',
]

df = pd.read_csv(DATA_PATH, usecols=USECOLS)
print(f'Raw shape: {df.shape}')


Raw shape: (50116, 23)


## 2. Filter the working sample (target cleaning)

Same logic as in `01_eda_andreja.ipynb`:

- Drop respondents who are **not eligible to vote** (`vote = 3`) — different population.
- Drop **non-response** (`vote ∈ {7, 8, 9}`) — uninformative.
- Recode `1 → 1` (voted), `2 → 0` (did not vote).

After this step, every downstream operation uses the filtered sample.


In [3]:
n_before = len(df)
df = df[df['vote'].isin([1, 2])].copy()
df['vote'] = (df['vote'] == 1).astype(int)
n_after = len(df)

print('=' * 64)
print(f'  WORKING SAMPLE: N = {n_after:,} rows  (dropped {n_before - n_after:,})')
print(f'  Class balance: voted = {df.vote.mean()*100:.1f}%, '
      f'did not vote = {(1-df.vote.mean())*100:.1f}%')
print('=' * 64)

WORKING_N = n_after


  WORKING SAMPLE: N = 46,470 rows  (dropped 3,646)
  Class balance: voted = 78.1%, did not vote = 21.9%


## 3. The cleaning helper

A single function that handles every cleaning step we need:

1. Replace ESS sentinel codes (`77 / 88 / 99`, `7 / 8 / 9`, `999`, etc.) with NaN.
2. Optionally **reverse-code** the scale (used for `polintr` so higher = more interest).
3. **Impute** the resulting NaNs — median for ordinal/continuous, mode for binary.
4. Optionally **recode binary 1/2 → 1/0** (for `gndr`, `clsprty`, `contplt`, `sgnptit`).

Calling it variable-by-variable below makes the cleaning steps fully transparent: each variable prints its sentinel count, imputation method, and post-cleaning mean.


In [4]:
def clean_variable(df, col, sentinels=None, impute='median',
                   reverse=False, recode_binary=False):
    """Clean one ESS variable. Returns the cleaned Series.

    Parameters
    ----------
    df : DataFrame (the working sample, N = WORKING_N)
    col : str — column name to clean
    sentinels : list of values to replace with NaN (e.g. [77, 88, 99])
    impute : 'median' (ordinal/continuous) or 'mode' (categorical/binary)
    reverse : True flips the scale so higher means "more" (used for polintr)
    recode_binary : True maps 1/2 → 1/0 (used for binary yes/no items)
    """
    s = df[col].copy()
    sentinels = sentinels or []

    # Step 1: sentinels -> NaN
    n_sentinel = int(s.isin(sentinels).sum()) if sentinels else 0
    if n_sentinel:
        s = s.where(~s.isin(sentinels), np.nan)

    # Step 2: reverse-code (only meaningful for ordinals; do BEFORE imputation)
    if reverse:
        mx = s.max(skipna=True)
        s = mx + 1 - s

    # Step 3: impute
    if impute == 'median':
        fill = s.median()
        s = s.fillna(fill)
    elif impute == 'mode':
        fill = s.mode().iloc[0]
        s = s.fillna(fill)
    else:
        fill = None  # leave NaNs in (used only for special cases)

    # Step 4: recode 1/2 -> 1/0 binary
    if recode_binary:
        s = (s == 1).astype(int)

    print(f'  {col:<9}  sentinels={n_sentinel:>5}  '
          f'impute={impute or "none":<6}  '
          f'mean={s.mean():.3f}  N={len(s):,}')
    return s


print('Helper defined.')


Helper defined.


## 4. Clean each variable, category by category

Same 7-category structure as `01_eda_andreja.ipynb` and the conceptual framework. Every cell prints what was done so the cleaning trail is fully visible.


### 4.1 Socio-demographic


In [6]:
print('Cleaning socio-demographic variables:')
df['agea'] = clean_variable(df, 'agea', sentinels=[999], impute='median')
df['gndr'] = clean_variable(df, 'gndr', sentinels=[9], impute='mode',
                            recode_binary=True)   # 1=Male, 2=Female -> 1/0
# cntry has no sentinels and we keep the 2-letter code as-is.
print(f'  cntry      no cleaning needed (no sentinels)  N={len(df):,}')


Cleaning socio-demographic variables:
  agea       sentinels=    0  impute=median  mean=53.055  N=46,470
  gndr       sentinels=    0  impute=mode    mean=0.459  N=46,470
  cntry      no cleaning needed (no sentinels)  N=46,470


### 4.2 Socio-economic resources

For `hinctnta` we additionally save a `was_missing` flag — income has a high refusal rate (~20%) and that missingness is itself informative.


In [8]:
print('Cleaning socio-economic variables:')
df['eduyrs'] = clean_variable(df, 'eduyrs', sentinels=[77, 88, 99], impute='median')
# Cap eduyrs at 99th percentile to handle data-entry outliers
p99 = df['eduyrs'].quantile(0.99)
df['eduyrs'] = df['eduyrs'].clip(upper=p99)
print(f'  eduyrs     capped at p99 = {p99:.0f} years')

# Income: save was_missing flag BEFORE imputing
df['hinctnta_was_missing'] = df['hinctnta'].isin([77, 88, 99]).astype(int)
df['hinctnta'] = clean_variable(df, 'hinctnta', sentinels=[77, 88, 99], impute='median')
print(f'  hinctnta_was_missing flag: '
      f'{df.hinctnta_was_missing.sum():,} respondents '
      f'({df.hinctnta_was_missing.mean()*100:.1f}%)')


Cleaning socio-economic variables:
  eduyrs     sentinels=  811  impute=median  mean=13.325  N=46,470
  eduyrs     capped at p99 = 24 years
  hinctnta   sentinels= 9198  impute=median  mean=5.321  N=46,470
  hinctnta_was_missing flag: 9,198 respondents (19.8%)


### 4.3 Political interest & ideology

`polintr` is reverse-coded so that **higher = more interest** (raw scale was 1=very interested … 4=not at all).
`lrscale` is kept raw, plus we add `lr_extreme = |lrscale - 5|` to capture ideological extremity (often more predictive than raw L–R).


In [12]:
print('Cleaning political-interest variables:')
df['polintr'] = clean_variable(df, 'polintr', sentinels=[7, 8, 9],
                               impute='median', reverse=True)
df['lrscale'] = clean_variable(df, 'lrscale', sentinels=[77, 88, 99], impute='median')
df['clsprty'] = clean_variable(df, 'clsprty', sentinels=[7, 8, 9],
                               impute='mode', recode_binary=True)

# Engineered: distance from the centre on the L-R scale
df['lr_extreme'] = (df['lrscale'] - 5).abs()
print(f'  lr_extreme  derived from lrscale: '
      f'mean={df.lr_extreme.mean():.2f}, range=[{df.lr_extreme.min()}, {df.lr_extreme.max()}]')


Cleaning political-interest variables:
  polintr    sentinels=   74  impute=median  mean=2.365  N=46,470
  lrscale    sentinels= 6470  impute=median  mean=5.068  N=46,470
  clsprty    sentinels= 1160  impute=mode    mean=0.433  N=46,470
  lr_extreme  derived from lrscale: mean=1.43, range=[0.0, 5.0]


### 4.4 Institutional trust (4 items, 0–10 scale)


In [13]:
print('Cleaning trust variables:')
for v in ['trstprl', 'trstplt', 'trstprt', 'trstlgl']:
    df[v] = clean_variable(df, v, sentinels=[77, 88, 99], impute='median')


Cleaning trust variables:
  trstprl    sentinels=  704  impute=median  mean=4.281  N=46,470
  trstplt    sentinels=  553  impute=median  mean=3.470  N=46,470
  trstprt    sentinels=  680  impute=median  mean=3.447  N=46,470
  trstlgl    sentinels=  919  impute=median  mean=5.210  N=46,470


### 4.5 Civic & political engagement


In [14]:
print('Cleaning civic-engagement variables:')
# Behavioural binaries (yes/no in last 12 months)
df['contplt'] = clean_variable(df, 'contplt', sentinels=[7, 8, 9],
                               impute='mode', recode_binary=True)
df['sgnptit'] = clean_variable(df, 'sgnptit', sentinels=[7, 8, 9],
                               impute='mode', recode_binary=True)

# Efficacy items (1-5 ordinal)
for v in ['cptppola', 'psppsgva', 'psppipla']:
    df[v] = clean_variable(df, v, sentinels=[7, 8, 9], impute='median')


Cleaning civic-engagement variables:
  contplt    sentinels=  241  impute=mode    mean=0.149  N=46,470
  sgnptit    sentinels=  232  impute=mode    mean=0.198  N=46,470
  cptppola   sentinels=  819  impute=median  mean=2.143  N=46,470
  psppsgva   sentinels=  813  impute=median  mean=2.159  N=46,470
  psppipla   sentinels=  732  impute=median  mean=2.102  N=46,470


### 4.6 Satisfaction with the system


In [15]:
print('Cleaning satisfaction variables:')
for v in ['stfdem', 'stfgov', 'stfeco']:
    df[v] = clean_variable(df, v, sentinels=[77, 88, 99], impute='median')


Cleaning satisfaction variables:
  stfdem     sentinels= 1454  impute=median  mean=4.880  N=46,470
  stfgov     sentinels= 1141  impute=median  mean=3.931  N=46,470
  stfeco     sentinels=  702  impute=median  mean=4.302  N=46,470


## 5. Build the composite indices (Set B only)

The EDA confirmed empirically:

- The four trust items correlate at r ≈ 0.57–0.88 (Spearman).
- The three efficacy items correlate at r ≈ 0.5–0.7.
- The three satisfaction items correlate at r ≈ 0.5–0.7.

This justifies replacing each block with its mean. The originals stay in `df` so Set A still has them — Set B is built by selecting the right columns at save time.


In [16]:
# Composite indices (added as new columns; originals are NOT deleted)
df['trust_index']        = df[['trstprl', 'trstplt', 'trstprt', 'trstlgl']].mean(axis=1)
df['efficacy_index']     = df[['cptppola', 'psppsgva', 'psppipla']].mean(axis=1)
df['satisfaction_index'] = df[['stfdem', 'stfgov', 'stfeco']].mean(axis=1)

print('Composite indices added:')
for c in ['trust_index', 'efficacy_index', 'satisfaction_index']:
    print(f'  {c:<20} mean={df[c].mean():.3f}  sd={df[c].std():.3f}')


Composite indices added:
  trust_index          mean=4.102  sd=2.255
  efficacy_index       mean=2.135  sd=0.807
  satisfaction_index   mean=4.371  sd=2.161


## 6. Sanity check the cleaned DataFrame

Three checks before we save anything:

1. **No NaNs** anywhere in the cleaned columns.
2. **No leftover sentinels** in any variable.
3. **N is still 46,470** — no rows were silently dropped.


In [17]:
# 1) NaN check
nan_cols = df.columns[df.isna().any()].tolist()
print(f'Columns with NaNs: {nan_cols if nan_cols else "None - clean."}')

# 2) Sentinel check (re-using the same sentinel map)
sent_map = {
    'agea':[999],'eduyrs':[77,88,99],'hinctnta':[77,88,99],
    'polintr':[7,8,9],'lrscale':[77,88,99],
    'trstprl':[77,88,99],'trstplt':[77,88,99],'trstprt':[77,88,99],'trstlgl':[77,88,99],
    'cptppola':[7,8,9],'psppsgva':[7,8,9],'psppipla':[7,8,9],
    'stfdem':[77,88,99],'stfgov':[77,88,99],'stfeco':[77,88,99],
    'gndr':[9],
}
remaining = {}
for c, codes in sent_map.items():
    n = int(df[c].isin(codes).sum())
    if n > 0:
        remaining[c] = n
print(f'\nLeftover sentinel codes: {remaining if remaining else "None - clean."}')

# 3) N check
assert len(df) == WORKING_N, f'N changed! {len(df)} vs {WORKING_N}'
print(f'\nN unchanged: {len(df):,} rows  ✓')


Columns with NaNs: None - clean.

Leftover sentinel codes: None - clean.

N unchanged: 46,470 rows  ✓


## 7. Build Feature Set A and Feature Set B

We keep `vote` as the rightmost column in both files (the modeling notebook expects this).


In [18]:
TARGET_COL = 'vote'

# --- Feature Set A: raw cleaned features (one row per individual variable) ---
SET_A_COLS = [
    # socio-demographic
    'agea', 'gndr', 'cntry',
    # socio-economic
    'eduyrs', 'hinctnta', 'hinctnta_was_missing',
    # political interest & ideology
    'polintr', 'lrscale', 'lr_extreme', 'clsprty',
    # institutional trust (4 items kept individually)
    'trstprl', 'trstplt', 'trstprt', 'trstlgl',
    # civic engagement (2 behaviours + 3 efficacy items)
    'contplt', 'sgnptit', 'cptppola', 'psppsgva', 'psppipla',
    # satisfaction (3 items kept individually)
    'stfdem', 'stfgov', 'stfeco',
]

# --- Feature Set B: composite indices replace the highly-correlated blocks ---
SET_B_COLS = [
    # socio-demographic
    'agea', 'gndr', 'cntry',
    # socio-economic
    'eduyrs', 'hinctnta', 'hinctnta_was_missing',
    # political interest & ideology
    'polintr', 'lrscale', 'lr_extreme', 'clsprty',
    # institutional trust  -> 1 composite
    'trust_index',
    # civic engagement: keep behaviours, replace efficacy with composite
    'contplt', 'sgnptit', 'efficacy_index',
    # satisfaction -> 1 composite
    'satisfaction_index',
]

set_A = df[SET_A_COLS + [TARGET_COL]].copy()
set_B = df[SET_B_COLS + [TARGET_COL]].copy()

print(f'Set A: {set_A.shape} ({len(SET_A_COLS)} features + target)')
print(f'Set B: {set_B.shape} ({len(SET_B_COLS)} features + target)')


Set A: (46470, 23) (22 features + target)
Set B: (46470, 16) (15 features + target)


## 8. Save both feature sets to disk

The modeling notebook loads from these CSVs — we don't re-run preprocessing inside the modeling notebook.


In [19]:
set_A.to_csv('feature_set_A.csv', index=False)
set_B.to_csv('feature_set_B.csv', index=False)

print('Saved:')
print(f'  feature_set_A.csv  ->  {set_A.shape[0]:,} rows × {set_A.shape[1]} cols')
print(f'  feature_set_B.csv  ->  {set_B.shape[0]:,} rows × {set_B.shape[1]} cols')

# Quick preview
print('\nFeature Set A preview:')
print(set_A.head(3))
print('\nFeature Set B preview:')
print(set_B.head(3))


Saved:
  feature_set_A.csv  ->  46,470 rows × 23 cols
  feature_set_B.csv  ->  46,470 rows × 16 cols

Feature Set A preview:
   agea  gndr cntry  eduyrs  hinctnta  hinctnta_was_missing  polintr  lrscale  \
0  65.0     1    AT    12.0       6.0                     0      4.0      5.0   
1  21.0     0    AT    14.0       1.0                     0      3.0      0.0   
2  53.0     0    AT    16.0       5.0                     0      3.0      3.0   

   lr_extreme  clsprty  ...  trstlgl  contplt  sgnptit  cptppola  psppsgva  \
0         0.0        1  ...      9.0        0        0       5.0       4.0   
1         5.0        1  ...      6.0        0        1       2.0       3.0   
2         2.0        1  ...      5.0        1        1       3.0       4.0   

   psppipla  stfdem  stfgov  stfeco  vote  
0       4.0     6.0     4.0     6.0     1  
1       3.0     7.0     5.0     2.0     1  
2       4.0     6.0     5.0     6.0     1  

[3 rows x 23 columns]

Feature Set B preview:
   agea  gndr 

## 9. Final summary table

A single table comparing what's in each feature set. This goes into the Word report's Methodology §3.5 (feature engineering).


In [20]:
# Build a side-by-side description
def describe_set(df_set, name):
    return pd.DataFrame({
        'feature': df_set.columns.drop(TARGET_COL),
        'set': name,
        'dtype': [str(df_set[c].dtype) for c in df_set.columns.drop(TARGET_COL)],
    })

summary = pd.concat([describe_set(set_A, 'A (raw)'),
                     describe_set(set_B, 'B (composite)')], ignore_index=True)
print(summary.groupby('set').agg(n_features=('feature', 'count'),
                                  features=('feature', lambda s: ', '.join(s))))


               n_features                                           features
set                                                                         
A (raw)                22  agea, gndr, cntry, eduyrs, hinctnta, hinctnta_...
B (composite)          15  agea, gndr, cntry, eduyrs, hinctnta, hinctnta_...


## 10. What's next

The next notebook (`03_modeling_andreja.ipynb`) will:

1. Load `feature_set_A.csv` and `feature_set_B.csv`.
2. One-hot encode `cntry` for the linear / NN models, ordinal-encode it for the trees.
3. Stratified 70 / 15 / 15 split.
4. Train **four models** on **both** feature sets:
   - Logistic Regression (baseline)
   - Random Forest
   - Gradient Boosting
   - Feed-Forward Neural Network
5. Tune hyperparameters with PR-AUC as the selection metric.
6. Save the trained models to disk.

Then `04_results_and_analysis_andreja.ipynb` produces the test-set metrics, ROC + PR curves, category-level permutation importance, SHAP plots, and per-country error analysis.
